# 14. 이진화·라벨링·외곽선

> 14번 강의 슬라이드의 여섯 코드 예제를 화면 순서대로 옮겼습니다.

강의 화면의 코드 순서와 파일명을 유지했습니다. 데이터 파일은 코드에 표시된 `./data` 또는 `../data` 상대 경로에 두세요.


## 서로 다른 전역 임계값 비교


In [ ]:
import sys
import cv2

src = cv2.imread("./data/cells.jpg", cv2.IMREAD_GRAYSCALE)

if src is None:
    print("Image load failed!")
    sys.exit()

_, dst1 = cv2.threshold(src, 100, 255, cv2.THRESH_BINARY)
_, dst2 = cv2.threshold(src, 210, 255, cv2.THRESH_BINARY)

cv2.imshow("src", src)
cv2.imshow("dst1", dst1)
cv2.imshow("dst2", dst2)
cv2.waitKey()
cv2.destroyAllWindows()


## Otsu 자동 임계값


In [ ]:
import sys
import cv2

src = cv2.imread("./data/rice.jpg", cv2.IMREAD_GRAYSCALE)

if src is None:
    print("Image load failed!")
    sys.exit()

th, dst = cv2.threshold(src, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
print("otsu's threshold:", th)

cv2.imshow("src", src)
cv2.imshow("dst", dst)
cv2.waitKey()
cv2.destroyAllWindows()


## 지역 적응형 이진화


In [ ]:
import sys
import cv2

src = cv2.imread("./data/sudoku.jpg", cv2.IMREAD_GRAYSCALE)

if src is None:
    print("Image load failed!")
    sys.exit()

bsize = 201
dst = cv2.adaptiveThreshold(
    src, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, bsize, 5
)

cv2.imshow("dst", dst)
cv2.imshow("srs", src)
cv2.namedWindow("dst")
cv2.waitKey()
cv2.destroyAllWindows()


## 작은 행렬의 연결 요소 라벨링


In [ ]:
import sys
import numpy as np
import cv2

mat = np.array(
    [
        [0, 0, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 1, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 1, 0],
        [0, 0, 0, 1, 1, 1, 1, 0],
        [0, 0, 1, 1, 0, 0, 1, 0],
        [0, 0, 1, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
    ],
    np.uint8,
)

cnt, labels = cv2.connectedComponents(mat)
print("sep:", mat, sep="\n")
print("cnt:", cnt)
print("labels:", labels, sep="\n")


## 라벨별 통계와 바운딩 박스


In [ ]:
import sys
import cv2

src = cv2.imread("./data/keyboard.jpg", cv2.IMREAD_GRAYSCALE)

if src is None:
    print("Image load failed!")
    sys.exit()

_, src_bin = cv2.threshold(src, 0, 255, cv2.THRESH_OTSU)
cnt, labels, stats, centroids = cv2.connectedComponentsWithStats(src_bin)
dst = cv2.cvtColor(src, cv2.COLOR_GRAY2BGR)

for i in range(1, cnt):
    (x, y, w, h, area) = stats[i]
    if area < 20:
        continue
    cv2.rectangle(dst, (x, y, w, h), (0, 255, 255))
    cv2.putText(dst, str(i), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 1, cv2.LINE_AA)

cv2.imshow("src", src)
cv2.imshow("src_bin", src_bin)
cv2.imshow("dst", dst)
cv2.waitKey()
cv2.destroyAllWindows()


## 외곽선으로 삼각형·사각형·원 판별


In [ ]:
import math
import cv2

def setLabel(img, pts, label):
    (x, y, w, h) = cv2.boundingRect(pts)
    pt1 = (x, y)
    pt2 = (x + w, y + h)
    cv2.rectangle(img, pt1, pt2, (0, 0, 255), 1)
    cv2.putText(img, label, pt1, cv2.FONT_HERSHEY_PLAIN, 1, (0, 0, 255))

def main():
    img = cv2.imread("./data/polygon.jpg", cv2.IMREAD_COLOR)
    if img is None:
        print("Image load failed!")
        return
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, img_bin = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(img_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    for pts in contours:
        if cv2.contourArea(pts) < 400:  # 너무 작으면 무시
            continue
        approx = cv2.approxPolyDP(pts, cv2.arcLength(pts, True) * 0.02, True)
        vtc = len(approx)
        if vtc == 3:
            setLabel(img, pts, "TRI")
        elif vtc == 4:
            setLabel(img, pts, "RECT")
        else:
            length = cv2.arcLength(pts, True)
            area = cv2.contourArea(pts)
            ratio = 4.0 * math.pi * area / (length * length)
            if ratio > 0.85:
                setLabel(img, pts, "CIR")
    cv2.imshow("img", img)
    cv2.waitKey()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()
